# Collisional coherence harvesting — concise tutorial

Two layers:

1. **Companion model** (analytic): V-system coherence drained onto ancillas; impedance-matched power.
2. **Spin-valve engine** (numeric): nonsecular leads + stroboscopic collisions; $(P_{\mathrm{el}}, P_{\mathrm{erg}})$.

Run from the repo root, or after `pip install -e .`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))
elif (ROOT.parent / "src").exists():
    sys.path.insert(0, str(ROOT.parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3})
print("imports ok")

## 1. Companion model: one collision

Closed-form map for doublet variables $(s, d, C)$ and the outgoing ancilla Bloch vector.

In [ ]:
from coherence_harvesting.companion.collision import (
    SystemState,
    analytic_system_map,
    analytic_ancilla_bloch,
    numerical_collision,
)

st = SystemState(s=0.5, d=0.0, C=0.2, p0=0.5)
theta, alpha = 0.4, 0.0  # exchange angle, ground-state ancilla

s2, d2, C2 = analytic_system_map(st.s, st.d, st.C, theta, alpha)
x, y, z = analytic_ancilla_bloch(st.s, st.d, st.C, theta, alpha)

rho_S, rho_A = numerical_collision(st.rho(), alpha, theta)
st_num = SystemState.from_rho(rho_S)

print(f"analytic: s={s2:.4f}, d={d2:.4f}, C={C2:.4f}")
print(f"numeric:  s={st_num.s:.4f}, d={st_num.d:.4f}, C={st_num.C:.4f}")
print(f"ancilla Bloch (x,y,z) = ({x:.4f}, {y:.4f}, {z:.4f})")
print(f"s invariant? {abs(s2 - st.s) < 1e-12}")

## 2. Fixed point and charging power

Bath regrows coherence between collisions. Too slow → little rate; too fast → depleted resource.
Optimum ≈ impedance matching $\Gamma_{\mathrm{ext}} = r(1-\cos\theta) \simeq \gamma_C$.

In [ ]:
from coherence_harvesting.companion.power import (
    CompanionParams,
    power_curve,
    optimize_power,
    impedance_matched_r,
    small_angle_bound,
)

params = CompanionParams(
    gamma_C=1.0,
    c0=0.25,
    s_ss=0.5,
    Delta=0.15,  # small matched gap (ergotropy scale)
    theta=0.35,
    alpha=0.0,
)

r, P = power_curve(params, exact=True)
r_opt, P_max = optimize_power(params)
r_imp = impedance_matched_r(
    params.gamma_C, params.theta, params.Delta / params.gamma_C
)
bound = small_angle_bound(
    params.gamma_C,
    params.Delta,
    params.c0,
    z_A=1.0,
    delta_over_gamma=params.Delta / params.gamma_C,
)

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(r / params.gamma_C, P, lw=2, label=r"$P_{\mathrm{erg}}(r)$")
ax.axvline(r_opt / params.gamma_C, color="C1", ls="--", label="numerical opt")
ax.axvline(r_imp / params.gamma_C, color="C2", ls=":", label="impedance match")
ax.axhline(bound, color="C3", ls="-.", label="small-angle bound")
ax.set_xscale("log")
ax.set_xlabel(r"$r / \gamma_C$")
ax.set_ylabel(r"$P_{\mathrm{erg}}$")
ax.set_title("Companion: charging power vs collision rate")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"r_opt = {r_opt:.4g},  P_max = {P_max:.4g}")
print(f"r_imp = {r_imp:.4g}  (should be close to r_opt)")

## 3. Accessibility of coherent ergotropy

Single-copy coherent ergotropy needs a phase reference. Two copies after energy dephasing activate only above a threshold (often **not** met in the weak harvesting regime).

In [ ]:
from coherence_harvesting.companion.ergotropy import (
    two_copy_accessible_work,
    qubit_ergotropy,
)

z, y = -0.9, 0.15  # weak ground-biased ancilla
W_total = qubit_ergotropy(z=z, x=0.0, y=y, Delta=params.Delta)
a, b = 0.5 * (1 - z), 0.5 * (1 + z)  # ground / excited
c_abs = 0.5 * abs(y)
W_acc2 = two_copy_accessible_work(a=a, b=b, c_abs=c_abs, Delta=params.Delta)

print(f"total ergotropy W   = {W_total:.4g}")
print(f"two-copy accessible = {W_acc2:.4g}")
print("(Often W_acc2 = 0 in the weak regime — report total as an upper bound.)")

## 4. Spin-valve engine (numeric)

Non-collinear leads regenerate spin coherence; collisions export it. Evaluate everything at the **stroboscopic fixed point**.

In [ ]:
from coherence_harvesting.transport.spin_valve import SpinValveParams
from coherence_harvesting.transport.engine import CollisionEngine
from coherence_harvesting.transport.nonsecular import (
    steady_state_leads,
    spin_coherence,
)

base = SpinValveParams(
    epsilon=0.0,
    Delta=0.3,
    gamma_L=1.0,
    gamma_R=1.0,
    p_L=0.7,
    p_R=0.7,
    theta_R=np.pi / 2,
    T_L=1.2,
    T_R=0.6,
    mu_L=0.4,
    mu_R=-0.4,
    g=8.0,
    tau=0.08,
    T_period=1.5,
    alpha=0.0,
)

rho_col = steady_state_leads(SpinValveParams(**{**base.__dict__, "theta_R": 0.0}))
rho_til = steady_state_leads(base)
print(f"|C| collinear = {abs(spin_coherence(rho_col)):.4g}")
print(f"|C| tilted    = {abs(spin_coherence(rho_til)):.4g}")

res = CollisionEngine(base).find_fixed_point(n_max=250)
print(f"fixed point: converged={res.converged}, n_iter={res.n_iter}")
print(f"  P_el  = {res.P_el:.4g}")
print(f"  P_erg = {res.P_erg:.4g}   (r * W_A)")
print(f"  W_A   = {res.W_A:.4g},  W_acc2 = {res.W_acc2:.4g}")
print(
    f"  |C|   = {abs(res.coherence):.4g},  "
    f"pops (p0,pu,pd) = {tuple(round(x, 4) for x in res.pops)}"
)

## 5. Quick tradeoff scan

Vary collision period $T$ (rate $r=1/T$) and collect both power channels.

In [ ]:
periods = np.geomspace(0.5, 6.0, 10)
rows = []
for T in periods:
    p = SpinValveParams(**{**base.__dict__, "T_period": float(T)})
    out = CollisionEngine(p).find_fixed_point(n_max=200, tol=1e-9)
    rows.append((1.0 / T, out.P_el, out.P_erg, abs(out.coherence)))

r_vals, Pel, Perg, Coh = map(np.array, zip(*rows))

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.5))
ax[0].plot(r_vals, Pel, "o-", label=r"$P_{\mathrm{el}}$")
ax[0].plot(r_vals, Perg, "s-", label=r"$P_{\mathrm{erg}}$")
ax[0].set_xscale("log")
ax[0].set_xlabel(r"$r = 1/T$")
ax[0].set_ylabel("power")
ax[0].set_title("Power channels vs collision rate")
ax[0].legend()

ax[1].plot(r_vals, Coh, "o-", color="C2")
ax[1].set_xscale("log")
ax[1].set_xlabel(r"$r = 1/T$")
ax[1].set_ylabel(r"$|\rho_{\uparrow\downarrow}|$")
ax[1].set_title("Dot coherence at fixed point")
plt.tight_layout()
plt.show()

## 6. Takeaways

| Point | Message |
|--------|---------|
| Companion optimum | Extraction rate matched to bath regrowth $\gamma_C$ |
| Finite gap $\Delta$ | Needed for battery energy; too large $\Rightarrow$ phase suppression |
| Accessibility | Total ergotropy is an upper bound; two copies often stay locked |
| Full engine | Numerics at the stroboscopic fixed point; frame as **resource partitioning** $(P_{\mathrm{el}}, P_{\mathrm{erg}})$, not collinear restoration |

Full derivation: `notes/coherence_harvesting_note.pdf`.  
Batch figures: `python scripts/generate_figures.py`.